# Bayesian Inference Review: SED Modeling of a Galaxy Spectrum

Inferring the stellar mass of a real galaxy from its SDSS spectrum, using the
[`provabgs`](https://github.com/changhoonhahn/provabgs) stellar population synthesis model.

> `pip install git+https://github.com/changhoonhahn/provabgs`

In [ ]:
import os, time
import numpy as np
import scipy.optimize as opt
import matplotlib.pyplot as plt
import emcee
import corner

from provabgs import models as Models
from provabgs import infer as Infer
%matplotlib inline
np.random.seed(0)

## get the SDSS spectrum
We'll download an SDSS spectrum and then mask its emission lines

In [ ]:
PLATE, MJD, FIBER = 2432, 54052, 327
CACHE = f'data/sdss_spec_full_{PLATE}_{MJD}_{FIBER}.npz'

def load_spectrum():
    if os.path.exists(CACHE):
        return np.load(CACHE)
    import requests
    from astropy.io import fits
    url = (f'https://dr17.sdss.org/sas/dr17/sdss/spectro/redux/26/spectra/lite/'
           f'{PLATE:04d}/spec-{PLATE:04d}-{MJD}-{FIBER:04d}.fits')
    open('tmp.fits','wb').write(requests.get(url, timeout=90).content)
    hd = fits.open('tmp.fits'); d = hd[1].data; sp = hd[2].data
    lam, fl, iv = 10**d['loglam'], d['flux'], d['ivar']
    g = iv > 0
    os.makedirs('data', exist_ok=True)
    np.savez_compressed(CACHE, wavelength=lam[g], flux=fl[g], ivar=iv[g],
                        z_pipeline=float(sp['Z'][0]), vdisp=float(sp['VDISP'][0]))
    return np.load(CACHE)

spec  = load_spectrum()
lam   = spec['wavelength']                 # Angstrom, vacuum
flux  = spec['flux']                       # 1e-17 erg/s/cm^2/A
sigma = 1/np.sqrt(spec['ivar'])
zred  = float(spec['z_pipeline'])
vdisp = float(spec['vdisp'])               # km/s, SDSS velocity dispersion

print(f'{len(lam)} pixels, {lam.min():.0f}-{lam.max():.0f} A')
print(f'z = {zred:.5f},  vdisp = {vdisp:.0f} km/s')
print(f'median S/N per pixel = {np.median(flux/sigma):.1f}')

Now lets mask the emission line. The SPS model describes starlight while the nebular emission lines come from ionized gas, and are not in this model.

In [ ]:
EMISSION_LINES = [3727.09, 3729.88,      # [OII]
                  4102.89, 4341.68,      # Hdelta, Hgamma
                  4862.68,               # Hbeta
                  4960.30, 5008.24,      # [OIII]
                  6302.05,               # [OI]
                  6549.86, 6564.61, 6585.27,   # [NII], Halpha
                  6718.29, 6732.68]      # [SII]
MASK_KMS = 600.
C_KMS = 299792.458

mask = np.zeros_like(lam, dtype=bool)
for l_rest in EMISSION_LINES:
    l_obs = l_rest*(1 + zred)
    mask |= np.abs(lam - l_obs) < l_obs*MASK_KMS/C_KMS
mask |= np.abs(lam - 5578.5) < 8.0        # sky line residual

use = ~mask
L, F, S = lam[use], flux[use], sigma[use]
print(f'masked {mask.sum()} pixels, fitting {use.sum()}')

## Always look at the data first


In [ ]:
plt.figure(figsize=(11, 3.8))
plt.plot(lam, flux, 'gray', lw=0.6, label='SDSS spectrum')
plt.plot(L, F, 'k', lw=1, label='SDSS spectrum (masked)')
ymax = np.percentile(flux, 99.8)
for l_rest in EMISSION_LINES:
    lo = l_rest*(1+zred)
    plt.axvspan(lo*(1-MASK_KMS/C_KMS), lo*(1+MASK_KMS/C_KMS), color='C3', alpha=0.18, lw=0)
plt.plot([], [], color='C3', alpha=0.4, lw=6, label='masked (nebular emission)')
plt.ylim(0, ymax*1.15); plt.xlabel(r'observed wavelength [$\AA$]')
plt.ylabel(r'flux [$10^{-17}$ erg s$^{-1}$ cm$^{-2}$ $\AA^{-1}$]')
plt.legend(frameon=False, fontsize=9, loc='upper right')
plt.tight_layout(); plt.show()

## The SED Model
A galaxy spectrum is the sum of light from all its stars. Stars of different mass, age, and metallicity have different spectra, so the shape of the spectrum encodes:

- how much stellar mass there is ($M_*$)
- when those stars formed — the **star formation history**
- what they are made of — the **metallicity history**
- how much dust sits between them and us

We can model the galaxy spectrum using stellar population synthesis (SPS). In particular, we can use the `provabgs` SPS model ([Hahn et al. (2023)](https://ui.adsabs.harvard.edu/abs/2022arXiv220201809H/abstract)), which has an emulator. It has the following free parameters:

| parameter | definition |
|---|---|
| `logmstar` | log₁₀ stellar mass |
| `beta1_sfh` … `beta4_sfh` | coefficients of 4 non-negative matrix factorization (NMF) star formation history bases that sum to 1 |
| `gamma1_zh`, `gamma2_zh` | coefficients of 2 NMF metallicity history bases |
| `dust1`, `dust2` | birth-cloud and diffuse ISM dust optical depths |
| `dust_index` | slope of the dust attenuation curve |

 For the purpose of this exercise you don't need to know the details. We'll also only focus on `logmstar`, `dust2`, and `dust_index`

In [ ]:
model = Models.NMF(burst=False, emulator=True)

prior = Infer.load_priors([
    Infer.UniformPrior(7., 13., label='sed'),                  # logmstar
    Infer.FlatDirichletPrior(4, label='sed'),                  # 4 SFH coefficients (sum to 1)
    Infer.LogUniformPrior(4.5e-5, 1.5e-2, label='sed'),        # gamma1_zh
    Infer.LogUniformPrior(4.5e-5, 1.5e-2, label='sed'),        # gamma2_zh
    Infer.UniformPrior(0., 3., label='sed'),                   # dust1
    Infer.UniformPrior(0., 3., label='sed'),                   # dust2
    Infer.UniformPrior(-2., 1., label='sed'),                  # dust_index
])

NDIM = prior.ndim_sampling
print(f'model parameters : {prior.ndim}')
print(f'sampling dimensions: {NDIM}')

In [ ]:
def model_flux(theta):
    '''SPS model flux 
    '''
    # there's some subtlety here with a transformation of the parameter space but it's unimportant for this exercise
    tt = prior.transform(np.atleast_2d(theta))[0] 
    _, fm = model.sed(tt, zred=zred, vdisp=vdisp, wavelength=L)
    return np.ravel(fm)

In [ ]:
plt.figure(figsize=(10, 3.8))
for i in range(8):
    theta_p = prior.sample()
    _flux = model_flux(theta_p)
    plt.plot(L, _flux/np.median(_flux), lw=0.9, alpha=0.8)
plt.xlabel(r'observed wavelength [$\AA$]'); plt.ylabel('flux / median')
plt.tight_layout(); plt.show()

## Set up your likelihood and posterior

In [ ]:
def log_prior(theta):
    return prior.lnPrior(theta) 

def log_likelihood(theta):
    # set up your Gaussian log-likelihood here using the function model_flux
    return 

def log_posterior(theta):
    # set up your posterior 
    return

## initialize the walkers

## Sample the posterior using `emcee`
* Run EnsembleSampler
* check burn in
* check convergence --- focus only on `logmstar`, `dust2` and `dust_index`, which correspond to [0, 7, 8] index of the parameter-space --- treat the rest as nuisance parameters
* plot the posterior of`logmstar`, `dust2` and `dust_index` using `corner`

# Compare with MPA-JHU

Calculate the 16, 50, 84th percentiles of the 1D marginal posterior of `logmstar`

*The synatx below assumes that `flat` is the flattened MCMC chain*

In [ ]:
logms  = flat[:, 0]

# derive 16, 50, 84th percentile


### How does our $\log M_*$ compare to the SDSS MPA-JHU catalog measurements
```
MPA_logMs = 9.67
```